# 13 — DenseNet OOF Test-Time Augmentation (No Retraining)

Mục tiêu: tận dụng các checkpoint đã train để kiểm tra TTA trên OOF mà **không train lại model**.

Dùng 2 finalist grayscale:
- seed 2026
- seed 3407

TTA medically conservative:
- original
- shift trái 4 px
- shift phải 4 px
- shift lên 4 px
- shift xuống 4 px

Không horizontal flip, không crop mạnh, không thay đổi intensity.

Notebook sẽ so:
1. no-TTA OOF
2. TTA từng seed
3. ensemble giữa hai seed sau TTA
4. threshold fine scan

Holdout vẫn niêm phong.


In [ ]:
print('>>> CELL 1 START: config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

from pathlib import Path
import json,time,gc
import numpy as np,pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
DIR26=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2'
DIR34=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407'
OUT=ROOT/'cv_outputs'/'densenet121_tta_oof'
OUT.mkdir(parents=True,exist_ok=True)

IMAGE_SIZE=224
BATCH_SIZE=16
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=True

TTA_VIEWS=[
    ('orig',0,0),
    ('left',-4,0),
    ('right',4,0),
    ('up',0,-4),
    ('down',0,4),
]

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('TTA views:',TTA_VIEWS,flush=True)


In [ ]:
print('>>> CELL 2 START: locked dev + grayscale cache', flush=True)
manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy().reset_index(drop=True)
holdout=manifest[manifest.split=='holdout'].copy().reset_index(drop=True)
assert len(dev)==4362 and len(holdout)==770
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
dev['cache_idx']=np.arange(len(dev))

for d in [DIR26,DIR34]:
    for f in range(5):
        p=d/f'fold{f}_best.pt'
        assert p.exists(),f'Missing checkpoint: {p}'

class PadToSquare:
    def __call__(self,img):
        w,h=img.size;s=max(w,h)
        l=(s-w)//2;r=s-w-l;t=(s-h)//2;b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
buf=[];t0=time.time()
for i,row in dev.iterrows():
    with Image.open(row.path) as im:
        im=im.convert('L').convert('RGB')
        im=pad(im).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        buf.append(np.asarray(im,dtype=np.uint8))
    if (i+1)%500==0 or i+1==len(dev):
        print(f'  cached {i+1}/{len(dev)}',flush=True)

DEV_CACHE=np.stack(buf)
del buf
print(f'Cache: {DEV_CACHE.shape} | {DEV_CACHE.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s',flush=True)
print('SEALED holdout:',len(holdout),flush=True)


In [ ]:
print('>>> CELL 3 START: helpers', flush=True)
MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}
to_tensor_norm=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class TTADataset(Dataset):
    def __init__(self,df,dx=0,dy=0):
        self.df=df.reset_index(drop=True);self.dx=dx;self.dy=dy
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        im=Image.fromarray(DEV_CACHE[int(r.cache_idx)])
        if self.dx!=0 or self.dy!=0:
            im=TF.affine(im,angle=0,translate=[self.dx,self.dy],scale=1.0,shear=[0.0,0.0],
                         interpolation=InterpolationMode.BILINEAR,fill=0)
        return to_tensor_norm(im),L2I[r.label],r.filename

def make_model():
    m=models.densenet121(weights=None)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

@torch.no_grad()
def infer(model,loader):
    model.eval();ys=[];ps=[];names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            z=model(x)
        p=torch.softmax(z.float(),1)[:,1]
        ys.extend(y.numpy());ps.extend(p.cpu().numpy());names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def score(y,p,t=.5,auc=None):
    y=np.asarray(y,int);p=np.asarray(p,float);pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y);se=tp/max(tp+fn,1);sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    if auc is None: auc=roc_auc_score(y,p)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),
            'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

def best_threshold(y,p):
    auc=float(roc_auc_score(y,p))
    ths=np.round(np.arange(.05,.951,.001),3)
    df=pd.DataFrame([score(y,p,float(t),auc) for t in ths])
    b=df.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    t=float(b.threshold)
    return t,score(y,p,t,auc),df


In [ ]:
print('>>> CELL 4 START: generate OOF TTA predictions', flush=True)

def generate_seed_oof(checkpoint_dir,seed_name):
    rows=[]
    for fold in range(5):
        print(f'\n{seed_name} | fold {fold}',flush=True)
        va=dev[dev.cv_fold==fold].copy().reset_index(drop=True)

        ck=torch.load(checkpoint_dir/f'fold{fold}_best.pt',map_location='cpu')
        model=make_model()
        model.load_state_dict(ck['state_dict'])

        view_probs=[]
        y_ref=None;name_ref=None
        for view,dx,dy in TTA_VIEWS:
            t0=time.time()
            dl=DataLoader(TTADataset(va,dx,dy),batch_size=BATCH_SIZE,shuffle=False,
                          num_workers=0,pin_memory=False)
            y,p,names=infer(model,dl)
            if y_ref is None:
                y_ref=y;name_ref=names
            else:
                assert np.array_equal(y_ref,y)
                assert name_ref==names
            view_probs.append(p)
            print(f'  {view}: {time.time()-t0:.1f}s',flush=True)

        arr=np.stack(view_probs,axis=1)
        p_orig=arr[:,0]
        p_tta=arr.mean(axis=1)

        part=pd.DataFrame({
            'filename':name_ref,
            'y_true':y_ref,
            'fold':fold,
            f'p_{seed_name}_orig':p_orig,
            f'p_{seed_name}_tta':p_tta,
        })
        rows.append(part)

        del model,ck,view_probs,arr
        gc.collect()
        if USE_AMP: torch.cuda.empty_cache()

    out=pd.concat(rows,ignore_index=True)
    assert len(out)==4362 and out.filename.nunique()==4362
    return out

o26=generate_seed_oof(DIR26,'g26')
o34=generate_seed_oof(DIR34,'g34')

merged=o26.merge(o34,on=['filename','y_true','fold'],validate='one_to_one')
assert len(merged)==4362
merged.to_csv(OUT/'tta_oof_predictions.csv',index=False)

print('>>> TTA OOF predictions saved',flush=True)


In [ ]:
print('>>> CELL 5 START: evaluate TTA + ensemble', flush=True)
y=merged.y_true.values
summary={}

for col in ['p_g26_orig','p_g26_tta','p_g34_orig','p_g34_tta']:
    t,m,scan=best_threshold(y,merged[col].values)
    summary[col]={'threshold':t,'metrics':m}
    scan.to_csv(OUT/f'{col}_threshold_scan.csv',index=False)

# Current best no-TTA weighted ensemble from notebook 12
p_current=.73*merged.p_g26_orig.values+.27*merged.p_g34_orig.values
t_cur,m_cur,scan_cur=best_threshold(y,p_current)
summary['current_073_027_no_tta']={'threshold':t_cur,'metrics':m_cur}
scan_cur.to_csv(OUT/'current_073_027_no_tta_scan.csv',index=False)

# Same weights after TTA
p_same=.73*merged.p_g26_tta.values+.27*merged.p_g34_tta.values
t_same,m_same,scan_same=best_threshold(y,p_same)
summary['same_weights_tta']={'threshold':t_same,'metrics':m_same}
scan_same.to_csv(OUT/'same_weights_tta_scan.csv',index=False)

# Re-optimize seed weight after TTA
rows=[]
ths=np.round(np.arange(.10,.901,.001),3)
for w26 in np.round(np.arange(0,1.001,.01),2):
    p=w26*merged.p_g26_tta.values+(1-w26)*merged.p_g34_tta.values
    auc=float(roc_auc_score(y,p))
    for t in ths:
        mm=score(y,p,float(t),auc)
        rows.append({'w26':float(w26),'w34':float(1-w26),**mm})

grid=pd.DataFrame(rows)
grid.to_csv(OUT/'tta_weight_threshold_grid.csv',index=False)
b=grid.sort_values(['P','w26','threshold'],ascending=[False,True,True]).iloc[0]
w26=float(b.w26);w34=float(b.w34);t=float(b.threshold)
p_best=w26*merged.p_g26_tta.values+w34*merged.p_g34_tta.values
best_m=score(y,p_best,t,float(roc_auc_score(y,p_best)))

fold_ps=[]
for f in range(5):
    mask=merged.fold.values==f
    fold_ps.append(score(y[mask],p_best[mask],t)['P'])

summary['best_tta_ensemble']={
    'w26':w26,'w34':w34,'threshold':t,'metrics':best_m,
    'fold_P_mean':float(np.mean(fold_ps)),
    'fold_P_std':float(np.std(fold_ps,ddof=1)),
    'fold_P_values':[float(x) for x in fold_ps]
}

summary['holdout_used']=False

with open(OUT/'tta_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2),flush=True)


## Decision rule

Current best trước TTA: **P ≈ 0.9870215**.

- Nếu TTA tăng rõ và không làm fold stability xấu đi nhiều → KEEP.
- Nếu gain rất nhỏ hoặc âm → bỏ TTA.
- Không retrain thêm chỉ để cứu một TTA yếu.
- Holdout vẫn chưa mở.
